# Dwarpal: fine-tune Indian plate OCR + detector (Colab GPU)

**Why:** the pretrained OCR (`cct-s-v2-global`, fast-plate-ocr) was trained on 65 countries but not India,
and Indian-LPR (the dataset named in the spec) is not public. This notebook:

1. **OCR**: synthetic Indian plates (`scripts/make_plate_ocr_dataset.py`), then *your* labeled crops if you
   have them (CSV `image_path,plate_text`), fine-tuned from the pretrained global model, exported to ONNX.
2. **Detector** (optional): YOLO26n fine-tuned on Indian plate boxes (Kaggle `kedarsai`, CC0) via
   `scripts/download_lpr.py`.
3. Downloads `dwarpal_plates.zip`; unzip into the repo and set the paths it prints in `config/settings.yaml`.

Runtime → Change runtime type → **GPU**.

In [ ]:
REPO_URL = "https://github.com/jay14090/dwarpal.git"
BRANCH = "main"
SYNTH_PLATES = 30000
REAL_CROPS_CSV = ""        # optional: Drive path to your labeled crops CSV (image_path,plate_text), images next to it
KAGGLE_USERNAME = ""       # optional, for the detector fine-tune
KAGGLE_KEY = ""
OCR_EPOCHS = 30
DET_EPOCHS = 60

In [ ]:
%cd /content
!rm -rf dwarpal && git clone --depth 1 -b $BRANCH $REPO_URL dwarpal
%cd /content/dwarpal
!pip -q install uv && uv sync --quiet
!pip -q install "fast-plate-ocr[train]" tensorflow
!git clone -q --depth 1 https://github.com/ankandrew/fast-plate-ocr /content/fpo
!wget -q -O /content/cct_s_v2_global.keras https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/cct_s_v2_global.keras

## 1. OCR fine-tune

In [ ]:
!uv run python scripts/make_plate_ocr_dataset.py --n $SYNTH_PLATES --out /content/plates_ocr
import pandas as pd, os
train = pd.read_csv("/content/plates_ocr/train.csv")
if REAL_CROPS_CSV:
    from google.colab import drive; drive.mount("/content/drive")
    real = pd.read_csv(REAL_CROPS_CSV)
    base = os.path.dirname(REAL_CROPS_CSV)
    real["image_path"] = [os.path.relpath(os.path.join(base, p), "/content/plates_ocr") for p in real["image_path"]]
    real["plate_text"] = real["plate_text"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
    real["plate_region"] = "Unknown"
    # oversample real crops so they are ~30% of an epoch
    reps = max(1, int(0.3 * len(train) / max(1, len(real))))
    train = pd.concat([train] + [real] * reps, ignore_index=True)
train.to_csv("/content/plates_ocr/train_mix.csv", index=False)
print(len(train), "training crops")

In [ ]:
!KERAS_BACKEND=tensorflow fast-plate-ocr train \
  --model-config-file /content/fpo/models/cct_s_v2.yaml \
  --plate-config-file /content/fpo/config/latin_plate_config_v2.yaml \
  --annotations /content/plates_ocr/train_mix.csv \
  --val-annotations /content/plates_ocr/val.csv \
  --weights-path /content/cct_s_v2_global.keras \
  --epochs $OCR_EPOCHS --batch-size 128 --lr 3e-4 \
  --output-dir /content/ocr_runs

In [ ]:
import glob
best = sorted(glob.glob("/content/ocr_runs/*/best.keras"))[-1]
!KERAS_BACKEND=tensorflow fast-plate-ocr export --model {best} --plate-config-file /content/fpo/config/latin_plate_config_v2.yaml --format onnx
!mkdir -p /content/out/models/plates && cp {best.replace('.keras', '.onnx')} /content/out/models/plates/ocr_india.onnx
!cp /content/fpo/config/latin_plate_config_v2.yaml /content/out/models/plates/ocr_india_config.yaml

## 2. Detector fine-tune (optional)

In [ ]:
import os
if KAGGLE_USERNAME:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = KAGGLE_USERNAME, KAGGLE_KEY
    !uv run python scripts/download_lpr.py --kaggle kedarsai/indian-license-plates-with-labels
    import random, shutil, pathlib
    src = pathlib.Path("data/raw/plates/kedarsai_indian-license-plates-with-labels")
    imgs = sorted((src / "images").iterdir()); random.Random(0).shuffle(imgs)
    for split, part in (("val", imgs[: len(imgs) // 10]), ("train", imgs[len(imgs) // 10 :])):
        for d in ("images", "labels"):
            (pathlib.Path("/content/plates_det") / split / d).mkdir(parents=True, exist_ok=True)
        for im in part:
            shutil.copy(im, f"/content/plates_det/{split}/images/{im.name}")
            shutil.copy(src / "labels" / f"{im.stem}.txt", f"/content/plates_det/{split}/labels/{im.stem}.txt")
    open("/content/plates_det/data.yaml", "w").write("path: /content/plates_det\ntrain: train/images\nval: val/images\nnames: {0: plate}\n")
    !uv run yolo detect train model=yolo26n.pt data=/content/plates_det/data.yaml epochs=$DET_EPOCHS imgsz=640 batch=32 project=/content/det_runs name=plates
    !cp /content/det_runs/plates/weights/best.pt /content/out/models/plates/yolo26n_plates.pt

## 3. Download and configure

In [ ]:
!cd /content/out && zip -qr /content/dwarpal_plates.zip models
print('''Unzip into the repo root, then in config/settings.yaml:
anpr:
  ocr:
    onnx_path: models/plates/ocr_india.onnx
    config_path: models/plates/ocr_india_config.yaml
  detector:            # only if you trained it
    backend: yolo
    weights: models/plates/yolo26n_plates.pt
Then measure: uv run python scripts/eval_plates.py --dataset-dir data/raw/plates_eval/<your set>''')
from google.colab import files
files.download("/content/dwarpal_plates.zip")